In [ ]:
# ============================================
# CELL 1: Install Dependencies
# ============================================
print("Installing dependencies...")

!pip install -q transformers accelerate bitsandbytes sentence-transformers
!pip install -q wandb pandas numpy scikit-learn datasets

print("Dependencies installed!")

In [ ]:
# # ============================================
# # CELL 4: Load Evaluation Dataset from HuggingFace
# # ============================================
# from datasets import load_dataset
# import pandas as pd
# import json

# print("Loading ParaDetox dataset from HuggingFace...")

# # Load dataset
# dataset = load_dataset("textdetox/multilingual_paradetox", split="en")

# # Filter for English only
# df = pd.DataFrame(dataset)

# print(f"Loaded {len(english_df)} English examples")
# print(f"Columns: {english_df.columns.tolist()}")
# print(f"\nSample:")
# print(english_df.head(2))

# # Create eval.jsonl format
# eval_data = []
# for idx, row in english_df.iterrows():
#     eval_entry = {
#         "id": idx,
#         "toxic": row['toxic_sentence'],
#         "polite": row['neutral_sentence'],
#         # "ref_tox": row['ref_tox'],
#         # "ref_neutral": row['neutral_score']
#     }
#     eval_data.append(eval_entry)

# # Save to eval.jsonl
# eval_file = "llm_data/eval.jsonl"
# with open(eval_file, 'w') as f:
#     for entry in eval_data:
#         f.write(json.dumps(entry) + '\n')

# print(f"\nSaved {len(eval_data)} examples to {eval_file}")

# # Load as DataFrame for experiments
# eval_df = pd.read_json(eval_file, lines=True)
# print(f"Evaluation dataset ready: {len(eval_df)} examples")

In [ ]:
# ============================================
# CELL: Create "Toxic -> Polite" Eval Set using Intel Classifier
# ============================================
from datasets import load_dataset
from transformers import pipeline
import pandas as pd
import json
import torch
from tqdm import tqdm

# 1. Setup the Intel Politeness Classifier
# ----------------------------------------
print("Loading Intel Politeness Classifier...")
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("text-classification", model="Intel/polite-guard", device=device)

# 2. Load the Raw Data
# ----------------------------------------
print("Loading ParaDetox dataset...")
dataset = load_dataset("textdetox/multilingual_paradetox", split="en")

print(f"Original dataset size: {len(dataset)}")

# 3. Filter Loop: Keep only "Polite" targets
# ----------------------------------------
eval_data = []
print("Filtering for politeness (this may take a moment)...")

# We iterate through the dataset and check the 'neutral_sentence'
# If Intel says it's polite, we keep it.
for idx, row in tqdm(enumerate(dataset), total=len(dataset)):

    candidate_polite = row['neutral_sentence']

    # Run the classifier on the neutral sentence
    # Truncate to 512 tokens to prevent errors with long sentences
    result = classifier(candidate_polite, truncation=True, max_length=512)[0]

    # LOGIC: Only keep if label is 'polite' AND confidence is high (> 0.75)
    if result['label'] == 'polite' and result['score'] > 0.70:
        eval_entry = {
            "id": len(eval_data), # New sequential ID
            "toxic": row['toxic_sentence'],
            "polite": candidate_polite,     # We use the verified polite text here
            "politeness_score": result['score'] # Keeping score for reference
        }
        eval_data.append(eval_entry)

# 4. Save to eval.jsonl
# ----------------------------------------
eval_file = "eval.jsonl"

# Ensure directory exists (optional, depending on your setup)
# import os
# os.makedirs(os.path.dirname(eval_file), exist_ok=True)

with open(eval_file, 'w') as f:
    for entry in eval_data:
        f.write(json.dumps(entry) + '\n')

# 5. Summary
# ----------------------------------------
print(f"\nProcessing Complete.")
print(f"Original rows: {len(dataset)}")
print(f"Filtered (Polite) rows: {len(eval_data)}")
print(f"Saved to: {eval_file}")

# Preview the dataframe
eval_df = pd.read_json(eval_file, lines=True)
print("\nSample of final dataset:")
print(eval_df[['toxic', 'polite', 'politeness_score']].head(3))

In [ ]:
# ============================================
# CELL 5: Weights & Biases Setup
# ============================================
import wandb

wandb.login()

wandb.init(
    project="detoxifyai-milestone2",
    name="few_shot_k=3",
    config={
        "model": "mistralai/Mistral-7B-Instruct-v0.2",
        "task": "toxic_rephrasing",
        "eval_dataset": "paradetox_english",
        "num_examples": len(eval_df)
    }
)

print("Weights & Biases ready!")

In [ ]:
# ============================================
# CELL 6: Load Mistral-7B Model
# ============================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Loading Mistral-7B-Instruct (4-bit quantized)...")

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Mistral-7B loaded!")
print(f"Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def generate_response(prompt, max_new_tokens=150, temperature=1.5):
    """Generate response from Mistral"""
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.replace(prompt, "").strip()
    return response

# Test
test_prompt = "[INST] Rephrase politely, keeping the same length: You're wrong [/INST]"
print(f"\n🧪 Test: {generate_response(test_prompt, max_new_tokens=50)}")

In [ ]:
# ============================================
# CELL 7: Define Prompt Strategy
# ============================================
# TODO: Modify this cell for each notebook variant

class PromptStrategy:
    def __init__(self, name):
        self.name = name

    def build_prompt(self, toxic_text):
        raise NotImplementedError

    def generate(self, prompt):
        return generate_response(prompt, max_new_tokens=150, temperature=0.7)


# ========================================
# CHANGE THIS SECTION FOR EACH NOTEBOOK
# ========================================

# OPTION 1: Zero-Shot (use this for notebook 1)
# class CurrentStrategy(PromptStrategy):
#     def __init__(self):
#         super().__init__("zero_shot")

#     def build_prompt(self, toxic_text):
#         return f"[INST] Rephrase the following text to be polite and constructive, keeping the same length: '{toxic_text}' [/INST]"


# OPTION 2: Few-Shot k=3 (use this for notebook 2)
class CurrentStrategy(PromptStrategy):
    def __init__(self):
        super().__init__("few_shot_k3")
        self.examples = [
            ("You're completely wrong.", "I see this differently."),
            ("That's a stupid idea.", "I have a different perspective."),
            ("You don't know anything.", "Perhaps we have different information.")
        ]

    def build_prompt(self, toxic_text):
        examples_text = "\n".join([f"Toxic: {t}\nPolite: {p}\n" for t, p in self.examples])
        return f"""[INST] Here are examples of rephrasing toxic text politely:

{examples_text}
Now rephrase: '{toxic_text}' [/INST]"""


# OPTION 3: Few-Shot k=5 (use this for notebook 3)
# class CurrentStrategy(PromptStrategy):
#     def __init__(self):
#         super().__init__("few_shot_k5")
#         self.examples = [
#             ("You're completely wrong.", "I see this differently."),
#             ("That's a stupid idea.", "I have a different perspective."),
#             ("You don't know anything.", "Perhaps we have different information."),
#             ("This is garbage.", "I think this could be improved."),
#             ("You're wasting my time.", "I'd prefer if we could be more efficient.")
#         ]
#
#     def build_prompt(self, toxic_text):
#         examples_text = "\n".join([f"Toxic: {t}\nPolite: {p}\n" for t, p in self.examples])
#         return f"""[INST] Here are examples of rephrasing toxic text politely:
#
# {examples_text}
# Now rephrase: '{toxic_text}' [/INST]"""


# OPTION 4: Chain-of-Thought (use this for notebook 4)
# class CurrentStrategy(PromptStrategy):
#     def __init__(self):
#         super().__init__("chain_of_thought")
#
#     def build_prompt(self, toxic_text):
#         return f"""[INST] Rephrase the following toxic text to be polite.
#
# Think step-by-step:
# 1. Identify what makes it toxic
# 2. Extract the core message
# 3. Rephrase politely while preserving meaning
#
# Toxic text: '{toxic_text}'
#
# Provide the polite rephrase: [/INST]"""


strategy = CurrentStrategy()
print(f"Strategy loaded: {strategy.name}")

In [ ]:
# ============================================
# CELL 8: Evaluation Metrics
# ============================================
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

def compute_similarity(text1, text2):
    """Compute cosine similarity"""
    emb1 = embedding_model.encode([text1])
    emb2 = embedding_model.encode([text2])
    return float(cosine_similarity(emb1, emb2)[0][0])

print("Evaluation functions ready!")

In [ ]:
# ============================================
# CELL 9: Run Experiment
# ============================================
from tqdm import tqdm
import time

print(f"\n{'='*60}")
print(f"Running Experiment: {strategy.name}")
print(f"{'='*60}\n")

results = {
    "strategy": strategy.name,
    "outputs": [],
    "similarities": [],
    "latencies": []
}

# Run on all eval examples
for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
    toxic_text = row['toxic']
    expected_polite = row['polite']

    # Build prompt
    prompt = strategy.build_prompt(toxic_text)

    # Generate with timing
    start_time = time.time()
    generated_polite = strategy.generate(prompt)
    latency = time.time() - start_time

    # Compute similarity
    similarity = compute_similarity(generated_polite, expected_polite)

    # Store results
    results["outputs"].append({
        "id": row['id'],
        "toxic": toxic_text,
        "expected": expected_polite,
        "generated": generated_polite,
        "similarity": similarity,
        "latency": latency
    })
    results["similarities"].append(similarity)
    results["latencies"].append(latency)

# Compute metrics
results["avg_similarity"] = np.mean(results["similarities"])
results["min_similarity"] = np.min(results["similarities"])
results["max_similarity"] = np.max(results["similarities"])
results["std_similarity"] = np.std(results["similarities"])
results["avg_latency"] = np.mean(results["latencies"])

print(f"\n📊 Results for {strategy.name}:")
print(f"   Avg Similarity: {results['avg_similarity']:.3f}")
print(f"   Min Similarity: {results['min_similarity']:.3f}")
print(f"   Max Similarity: {results['max_similarity']:.3f}")
print(f"   Std Similarity: {results['std_similarity']:.3f}")
print(f"   Avg Latency: {results['avg_latency']:.3f}s")

# Log to W&B
wandb.log({
    "avg_similarity": results["avg_similarity"],
    "min_similarity": results["min_similarity"],
    "max_similarity": results["max_similarity"],
    "std_similarity": results["std_similarity"],
    "avg_latency": results["avg_latency"]
})

print("\nExperiment complete!")

In [ ]:
# ============================================
# CELL 10: Save Results
# ============================================

# Save detailed outputs
output_file = f"{strategy.name}_outputs.json"
with open(output_file, 'w') as f:
    json.dump(results["outputs"], f, indent=2)

print(f"Saved detailed outputs to {output_file}")

# Save metrics summary
metrics = {
    "strategy": strategy.name,
    "avg_similarity": results["avg_similarity"],
    "min_similarity": results["min_similarity"],
    "max_similarity": results["max_similarity"],
    "std_similarity": results["std_similarity"],
    "avg_latency": results["avg_latency"],
    "num_examples": len(results["outputs"])
}

metrics_file = f"{strategy.name}_metrics.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {metrics_file}")

# Show sample outputs
print("\nSample Outputs:")
for i in range(min(3, len(results["outputs"]))):
    ex = results["outputs"][i]
    print(f"\n--- Example {i+1} ---")
    print(f"Toxic:     {ex['toxic'][:80]}...")
    print(f"Expected:  {ex['expected'][:80]}...")
    print(f"Generated: {ex['generated'][:80]}...")
    print(f"Similarity: {ex['similarity']:.3f}")


wandb.finish()
print("\n✅ All results saved!")